# **Lab 01** – The VRAM Constraint and 4D Tensor Engineering

## Overview

**Objective:** This lab introduces the fundamental engineering constraints of 3D video architectures. You will bypass abstracted frameworks to manually control temporal subsampling, spatial dimensionality reduction, and hardware memory limits on an NVIDIA T4 GPU.

Students will learn to: 
- Read and manipulate raw 4D video tensors.
- Reorder memory structures to align with PyTorch 3D convolutional mathematical requirements.
- Calculate VRAM requirements before running any code.
- Identify and fix memory leaks in training loops.
- Prove the necessity of temporal data augmentation.

**Data to use**
You will use a curated 33-video micro-dataset extracted from UCF101. 
In your Kaggle notebook, click **Add Data**, and search for the URL slug:
`uvigo-video-understanding-lab-01`

## Rules of engagement

This notebook contains **broken, sabotaged, or incomplete code**.  
Your job is to **diagnose, fix, and justify** — not to write pipelines from scratch.

Every task follows the same structure:

1. **Read** the sabotaged cell carefully.
2. **Identify** the failure mode before touching the code.
3. **Fix** the minimum number of lines required.
4. **Answer** the analytical question in the Markdown cell that follows.

> **Anti-shortcut policy.** Running a cell and seeing it produce *some* output is not evidence of correctness. You must verify against the explicit acceptance criteria stated in each task. A pipeline that silently returns a zero tensor is wrong, even if it does not crash.

---

## Environment setup

Run the cell below first. It will refuse to continue if the GPU is not attached.

In [ ]:
# =========================
# Setup and Dependencies
# =========================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import cv2
import numpy as np
import os, glob, random, time
import matplotlib.pyplot as plt

# ── Video loading helper (OpenCV) ──────────────────────────────────────────────
def load_video(path: str) -> torch.Tensor:
    cap    = cv2.VideoCapture(path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return torch.from_numpy(np.stack(frames))  # [T, H, W, C], uint8

# ── Hardware gate ──────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise SystemError(
        'T4 GPU not detected. Go to Settings -> Accelerator -> GPU T4 x1 and restart.'
    )

gpu     = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024**3
print(f'GPU   : {gpu.name}')
print(f'VRAM  : {vram_gb:.2f} GB')
print(f'PyTorch: {torch.__version__}')


## **Task 1** – The Silent Axis Crime

A junior engineer wrote the permutation below after extracting frames with a standard image loader.  
The code **runs without errors** and even passes through a `v2.Resize`. That is the problem.

**Step 1.** Before changing anything, answer in the Markdown cell below:
- What shape does `pytorch_tensor` currently have?
- What shape does `nn.Conv3d` expect its input to be, for a batch of `B` clips?
- Why does the misalignment survive `v2.Resize` silently?

**Step 2.** Fix the single `permute` call so `final_video` has shape `[C, T, H, W] = [3, 4, 224, 224]`.  
Do not change anything else.

**Acceptance criterion:** The assertion at the bottom of the cell must pass.

In [ ]:
# Raw video simulation — decoder output shape: [T, H, W, C]
raw_video_tensor = torch.randint(0, 255, (4, 320, 240, 3), dtype=torch.uint8)

# ── SABOTAGE: one number in the permute tuple is wrong ────────────────────────
pytorch_tensor = raw_video_tensor.permute(0, 3, 1, 2)   # <-- fix this line

resize      = v2.Resize((224, 224), antialias=True)
final_video = resize(pytorch_tensor)

print(f'Shape after permute+resize : {final_video.shape}')

# Acceptance criterion — do not modify
assert final_video.shape == (3, 4, 224, 224), (
    f'FAIL: expected (3, 4, 224, 224), got {final_video.shape}'
)
print('PASS')

### Task 1 — Analysis

*Replace this text with your answers.*

**Q1.** What shape did the sabotaged permute produce, and why did it not crash?

**Q2.** If you fed that malformed tensor into `nn.Conv3d(in_channels=3, ...)`, which dimension would the convolution mistakenly treat as the channel axis? What would the runtime error message tell you?

---
## Task 2 — Three Defects, One Dataset

The `VideoDataset` below has **three independent defects**. They produce distinct failure modes, each caught by a different assertion.

**Before touching the code**, read every line and identify all three defects. Write them in the Markdown cell below *before* you fix anything — this is the diagnostic step.

Fix all three. The three assertions at the bottom must pass on every video in the dataset.

In [ ]:
DATASET_PATH = '/kaggle/input/datasets/ivnrodrguezconde/uvigo-video-understanding-lab-01/uvigo-video-ucf101-micro'

class VideoDataset(Dataset):
    def __init__(self, data_path, num_frames=16, stride=2):
        class_names = sorted([
            d for d in os.listdir(data_path)
            if os.path.isdir(os.path.join(data_path, d))
        ])
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.video_paths, self.labels = [], []
        for cls in class_names:
            for vp in glob.glob(os.path.join(data_path, cls, '*.avi')):
                self.video_paths.append(vp)
                self.labels.append(self.class_to_idx[cls])
        self.num_frames = num_frames
        self.stride     = stride
        self.transform  = v2.Resize((224, 224), antialias=True)

    def __len__(self): return len(self.video_paths)

    def __getitem__(self, idx):
        video_tensor = load_video(self.video_paths[idx])  # dtype: uint8

        total_frames = video_tensor.shape[0]
        required     = self.num_frames * self.stride

        indices = list(range(0, required, 1))

        subsampled = video_tensor[indices]

        subsampled = subsampled.permute(0, 3, 1, 2)

        final_tensor = (subsampled / 255).to(torch.uint8)

        return self.transform(final_tensor), torch.tensor(self.labels[idx])


# ── Verification — do not modify ──────────────────────────────────────────────
ds = VideoDataset(DATASET_PATH)
for i in range(len(ds)):
    clip, label = ds[i]
    assert clip.shape == (3, 16, 224, 224), f'[{i}] shape: {clip.shape}'
    assert clip.dtype == torch.float32,     f'[{i}] dtype: {clip.dtype}'
    assert clip.mean() > 0.01,              f'[{i}] Darkness Bug active: mean={clip.mean():.4f}'

print(f'All {len(ds)} clips passed.')



### Task 2 — Defect Report

*Replace this text with your pre-fix diagnosis.*

**Defect A — location and root cause:**

**Defect B — location and root cause:**  
*(Note: if you find an axis defect here that appears to contradict Task 1, explain why each permutation is correct in its own context.)*

**Defect C — location and root cause:**

**Q.** Defect C calls `.to(torch.uint8)` after dividing. What does this cast do to float values in the range $[0, 1]$, and why does the dtype assertion catch it while the mean assertion might not?

---
## Task 3 — VRAM Arithmetic Before You Write a Line of Code

A model runs out of memory. The engineer's first instinct is to reduce batch size.  
That is the wrong first instinct. The correct first instinct is to **calculate**.

The cell below contains a sabotaged baseline formula and three incomplete reduction steps. Fix the baseline and complete the reductions. The assertions are your acceptance criteria — fix your formulas until they pass, not the other way around.

| Parameter | Value |
|---|---|
| Batch size $B$ | 8 clips |
| Frames per clip $T$ | 16 |
| Resolution | $224 \times 224$ |
| Channels $C$ | 3 (RGB) |
| Data type | `float32` (4 bytes) |

Then answer the analytical questions below using the numbers your code produces.

In [ ]:
# ── Step 1: baseline tensor VRAM ─────────────────────────────────────────────
B, C, T, H, W = 8, 3, 16, 224, 224
BYTES_PER_FLOAT32 = 4

# SABOTAGE: the formula below is wrong
# Fix it so it correctly computes the VRAM bytes.
VRAM_bytes = B * C * H * W * BYTES_PER_FLOAT32   # <-- fix this line

print(f'Baseline  : {VRAM_bytes:,} bytes  |  {VRAM_bytes/1024**2:.1f} MB  |  {VRAM_bytes/1024**3:.3f} GB')

VRAM_MB = VRAM_bytes / 1024**2
VRAM_GB = VRAM_bytes / 1024**3
print(f'Baseline  : {VRAM_bytes:,} bytes  |  {VRAM_MB:.1f} MB  |  {VRAM_GB:.3f} GB')

# ── Reduction 1: temporal stride k=2 ────────────────────────────────────────
T_r = # TODO: compute the number of frames after applying stride k=2
bytes_r1 = # TODO: compute VRAM after keeping every other frame
assert bytes_r1 == VRAM_bytes // 2, f'R1 failed: {bytes_r1} != {VRAM_bytes//2}'
print(f'After T-stride k=2 : {bytes_r1/1024**2:.1f} MB')

# ── Reduction 2: spatial resize to 112x112 ───────────────────────────────────
bytes_r2 = # TODO: compute VRAM after the spatial resize
assert bytes_r2 == bytes_r1 // 4, f'R2 failed: {bytes_r2} != {bytes_r1//4}'
print(f'After 112x112      : {bytes_r2/1024**2:.1f} MB')

# ── Reduction 3: cast to float16 ─────────────────────────────────────────────
bytes_r3 =  # TODO: compute VRAM after casting to float16, starting from bytes_r2
assert bytes_r3 == bytes_r2 // 2, f'R3 failed: {bytes_r3} != {bytes_r2//2}'
print(f'After float16      : {bytes_r3/1024**2:.1f} MB')

print(f'Total reduction factor : {VRAM_bytes / bytes_r3:.1f}x')

# ── Q2: hardware fit check ───────────────────────────────────────────────────
T4_VRAM_GB = 16.0
# TODO: compute total training VRAM in GB (assume 3x input tensor size)
#       and print whether the baseline configuration fits on a T4

# ── Q3: absolute savings per reduction ──────────────────────────────────────
# TODO: compute the absolute byte saving of each reduction
#       and print which single reduction saves the most bytes

### Task 3 — Analysis

**Q1.** The sabotaged formula excluded $T$. By what multiplicative factor did it underestimate the true tensor size? Use the value your code printed.

**Q2.** At the baseline configuration, does a full training step fit on the T4? Show the arithmetic using the values your code produced.

**Q3.** Which single reduction gives the largest absolute byte saving? Justify with the numbers your code printed, and explain geometrically why the spatial resize saves more than the temporal stride, given that the resize halves both spatial dimensions while the stride halves only the temporal one.

---
## Task 4 — The Ghost Gradient Leak

The training loop below contains one line that causes unbounded memory growth across iterations.
The model is correct. The training step is correct. The leak is in the accumulation logic.

**Part A — Diagnosis.**
Read the training loop carefully. Identify the line responsible for the memory growth and write
your explanation in the Markdown cell below *before* making any changes. Your explanation must
describe the mechanism, not just name the line.

**Part B — Fix and verify.**
Fix the line and run both the sabotaged and clean versions. The VRAM after 10 iterations must
be stable in the clean version — i.e. equal to the VRAM before the loop.

**Part C — Analysis.**
Answer the questions in the Markdown cell below.

In [ ]:
class Simple3DCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 8 * 112 * 112, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.head(self.conv(x))

model     = Simple3DCNN().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def run_training_loop(label='TEST', n_iterations=10):
    torch.cuda.empty_cache()
    history = []
    print(f'--- {label} ---')
    print(f'VRAM before loop : {torch.cuda.memory_allocated()/1024**2:.0f} MB')

    for i in range(n_iterations):
        data   = torch.randn(8, 3, 16, 224, 224).cuda()
        target = torch.randint(0, 2, (8,)).cuda()
        optimizer.zero_grad()
        output = model(data)
        loss   = criterion(output, target)
        loss.backward()
        optimizer.step()

        history.append(output)

    print(f'VRAM after {n_iterations} iterations : {torch.cuda.memory_allocated()/1024**2:.0f} MB')
    print(f'Tensors retained in history : {len(history)}')
    return history


# Part A: read the training loop carefully.
# Identify the line that causes unbounded memory growth and explain the mechanism
# in the Markdown cell below BEFORE making any changes.

# Part B: run the sabotaged version first, then fix it and run again.
history = run_training_loop('LEAKY (sabotaged)')

# After fixing, uncomment and run:
# history_clean = run_training_loop('CLEAN (fixed)')

### Task 4 — Analysis

*Write your Part A diagnosis here before touching the code.*

**Diagnosis:** identify the line and explain the mechanism in one paragraph.

---

*Answer the following after completing Parts A and B.*

**Q1.** Explain in one sentence why appending `output` retains the computational graph,
while appending `output.detach()` does not.

**Q2.** Each iteration appends one `output` tensor of shape `[B, 2]`along with the full computational graph. Explain why the memory retained per iteration scales with batch size `B`, accounting for both the output tensor and the intermediate activations stored for backpropagation.

**Q3.** The `torch.cuda.empty_cache()` call is absent from the fixed loop but was present in many online examples you may have seen. Explain why cache clearing cannot reclaim memory that is still referenced by a Python list.

---
## Task 5 — The Memorisation Trap and the Jitter Fix

A 3D CNN trained without temporal augmentation will memorise specific frame sequences
rather than learning the underlying action. The fix is temporal jitter: randomising the
start frame on every call to `__getitem__` so the model never sees the exact same
4D tensor twice.

**Part A — Refactor `VideoDataset`.**
Add a `jitter: bool = True` parameter to `VideoDataset.__init__`. When `jitter=True`,
select a random valid start frame in `__getitem__` instead of always starting at frame 0.

> Write the formula for the maximum valid start index $s_{max}$ as a comment directly
> above the `random.randint` call. The formula must involve `total_frames`, `num_frames`,
> and `stride`.

**Part B — Verify the fix.**
Run the cell below. With jitter enabled, two calls to `ds[0]` must return different tensors.
With jitter disabled, two calls must return identical tensors. Both assertions must pass.

In [ ]:
# ── Part B: verify jitter behaviour ──────────────────────────────────────────
ds_jitter = VideoDataset(DATASET_PATH, jitter=True)
clip1, _  = ds_jitter[0]
clip2, _  = ds_jitter[0]

ds_fixed  = VideoDataset(DATASET_PATH, jitter=False)
clip3, _  = ds_fixed[0]
clip4, _  = ds_fixed[0]

print(f"Jitter ON  — same clip called twice, tensors differ : {not torch.equal(clip1, clip2)}")
print(f"Jitter OFF — same clip called twice, tensors equal  : {torch.equal(clip3, clip4)}")

assert not torch.equal(clip1, clip2), "FAIL: jitter=True returned identical tensors"
assert torch.equal(clip3, clip4),     "FAIL: jitter=False returned different tensors"
print("PASS")

### Task 5 — Analysis

**Q1.** Derive the formula for the maximum valid start index $s_{max}$ given total frames $F$,
clip length $N$, and stride $k$. Prove that any $s \leq s_{max}$ guarantees no out-of-bounds access.

**Q2.** A 3D CNN trained without jitter on a small dataset will memorise specific
spatial-temporal patterns rather than learning the underlying action. Describe concretely
what the 3D convolutional kernels have learned to detect after seeing the same
4D tensor for 20 epochs.

**Q3.** If a video has $F=150$ frames, $N=16$, and $k=2$, how many distinct clip
start positions can jitter produce from this single video?

---
## Task 6 — Final Report

Answer each question precisely. Show arithmetic where requested.
Unsupported claims receive no credit.

---

### 6.1 The Bug Audit

The notebook contains four sabotages across Tasks 1, 2, and 4. For each one state:
- The **task and line** where the defect lives.
- The **first principle** it violated (e.g. axis semantics, integer arithmetic,
reference counting, dtype propagation, ...).
- The **minimal fix** (one line or fewer).
- The **assertion** that catches it and why that assertion is sensitive to this specific defect.

---

### 6.2 Activation Memory

A single forward pass through `Simple3DCNN` at $B=8$, float32,
input `[8, 3, 16, 224, 224]` produces the following intermediate tensors:

| Layer | Output shape |
|---|---|
| `Conv3d(3→16, k=3, p=1)` | `[8, 16, 16, 224, 224]` |
| `ReLU` | same |
| `MaxPool3d(2)` | `[8, 16, 8, 112, 112]` |
| `Flatten` | `[8, 1605632]` |
| `Linear(→128)` | `[8, 128]` |
| `Linear(→2)` | `[8, 2]` |

Calculate the total activation memory in **MB** required to store all of these tensors
simultaneously during backpropagation (float32, 4 bytes per element).

---

### 6.3 The MLP Paradox

A naive `nn.Linear` layer receives a flattened video tensor of size $3 \times 16 \times 224 \times 224$.

1. Calculate the exact parameter count for `nn.Linear(in, 256)` including bias.
2. Calculate the weight memory in **MB** (float32).
3. Explain why this makes MLPs *weight-bound* on video data while 3D CNNs are *activation-bound*.

---

### 6.4 The Stride Defect

Defect A in Task 2 uses `range(0, required, 1)` instead of `range(0, required, self.stride)`.
For a video with `total_frames=60`, `num_frames=16`, and `stride=2`:

1. Compute the exact shape of `subsampled` before and after the fix.
2. Explain why the shape assertion `clip.shape == (3, 16, 224, 224)` catches this defect.
3. Explain why a visual inspection of the returned frames might not catch it —
   i.e. what would the frames look like and why would they appear plausible?